### Importar librerías

In [ ]:
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt

from tensorflow.keras.utils import plot_model
from tensorflow.keras.callbacks import EarlyStopping

### Cargar el dataset MNIST (dígitos escritos a mano)

In [ ]:
# Importar el dataset
digits_data = keras.datasets.mnist

# Cargar datos de entrenamiento y prueba
(train_images, train_labels), (test_images, test_labels) = digits_data.load_data()

### Crear nombres de las clases

In [ ]:
class_names = [str(i) for i in range(10)]
class_names

### Mostrar una imagen de ejemplo (tamaño original 28x28)

In [ ]:
plt.imshow(train_images[7], cmap="gray")
plt.grid(False)
plt.axis("off")
plt.show()

### Verificar el tamaño del dataset original

In [ ]:
train_images.shape, test_images.shape

### Distribución de clases

In [ ]:
df = pd.DataFrame(np.unique(train_labels, return_counts=True)).T
df.columns = ["Label", "Count"]
df

### Redimensionar las imágenes a 16x16

El modelo se va a usar después en una GUI donde la persona dibuja el dígito en un canvas de 16x16 pixeles,
así que redimensionamos el dataset de 28x28 a 16x16 para que la entrada del modelo coincida con la entrada real que vamos a mandarle desde la web.

In [ ]:
IMG_SIZE = 16

def resize_images(images):
    images = images.astype("float32")
    images = images[..., np.newaxis]  # (N, 28, 28, 1)
    images = tf.image.resize(images, (IMG_SIZE, IMG_SIZE), method="bilinear")
    images = images.numpy().squeeze(-1)  # (N, 16, 16)
    return images

train_images = resize_images(train_images)
test_images = resize_images(test_images)

train_images.shape, test_images.shape

### Mostrar la misma imagen ya en 16x16 (para comparar contra la de 28x28)

In [ ]:
plt.imshow(train_images[7], cmap="gray")
plt.grid(False)
plt.axis("off")
plt.show()

### Mostrar las primeras 16 imágenes en 16x16

In [ ]:
plt.figure(figsize=(9,9))

for i in range(16):
    plt.subplot(4,4,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i], cmap="gray")
    plt.title(class_names[train_labels[i]])

plt.show()

### Normalizar las imágenes

Escalamos de 0-255 a 0-1, igual que en escala de grises normalizada.

In [ ]:
train_images = train_images / 255.0
test_images = test_images / 255.0

train_images.min(), train_images.max()

### Crear la red neuronal

- Capa de entrada: `Flatten` con la forma que depende del número de pixeles (16x16 = 256 entradas).
- Capas ocultas: `Dense` con activación `relu` (cantidad de capas y neuronas libre).
- Capa de salida: `Dense(10, activation="softmax")`, una neurona por cada dígito (0-9).

In [ ]:
model = keras.Sequential([
    keras.layers.Flatten(input_shape=(IMG_SIZE, IMG_SIZE)),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

In [ ]:
model.summary()

### Compilar el modelo

Como es clasificación multiclase con etiquetas como enteros (0-9, no one-hot), usamos `sparse_categorical_crossentropy`.
El optimizador es `Adam` (aquí se deja el learning rate libre para ajustarlo).

In [ ]:
optimizer = keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Ver resumen del modelo

In [ ]:
model.summary()

### Dibujar arquitectura

In [ ]:
plot_model(
    model,
    to_file="model_plot.png",
    show_shapes=True,
    show_layer_names=True
)

### Configurar Early Stopping

Detiene el entrenamiento si la pérdida de validación deja de mejorar, y restaura los mejores pesos encontrados.

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

### Entrenar el modelo

In [ ]:
history = model.fit(
    train_images,
    train_labels,
    epochs=50,
    validation_split=0.2,
    callbacks=[early_stop]
)

### Graficar la precisión

In [ ]:
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Model Accuracy")
plt.ylabel("Accuracy")
plt.xlabel("Epoch")
plt.legend(["Train", "Validation"], loc="lower right")
plt.show()

### Graficar la pérdida

In [ ]:
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("Model Loss")
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.legend(["Train", "Validation"], loc="upper right")
plt.show()

### Evaluar el modelo

In [ ]:
test_loss, test_acc = model.evaluate(test_images, test_labels)

print("Test Accuracy:", test_acc)

### Realizar predicciones

In [ ]:
predictions = model.predict(test_images)

### Ver las posibilidades de la primera imagen

In [ ]:
predictions[0].round(2)

### Comparar predicción con etiqueta real

In [ ]:
print("Predicción:", np.argmax(predictions[0]))
print("Etiqueta real:", test_labels[0])

### Mostrar 16 predicciones aleatorias

In [ ]:
figure = plt.figure(figsize=(9,9))

indices = np.random.choice(
    test_images.shape[0],
    size=16,
    replace=False
)

for i, index in enumerate(indices):

    ax = figure.add_subplot(4,4,i+1)

    ax.set_xticks([])
    ax.set_yticks([])

    ax.imshow(test_images[index], cmap="gray")

    predict_index = np.argmax(predictions[index])
    true_index = test_labels[index]

    ax.set_title(
        f"{class_names[predict_index]} ({class_names[true_index]})",
        color="green" if predict_index == true_index else "red"
    )

plt.tight_layout()
plt.show()

### Guardar el modelo en formato .h5

Este archivo `.h5` es el que vamos a subir a GitHub junto con `app.py` y el resto de archivos para desplegar en Render.

In [ ]:
model.save("digit_model.h5")
print("Modelo guardado como digit_model.h5")

### Guardar las clases (labels) en un archivo aparte

Guardamos también un pequeño archivo con las clases y el tamaño de imagen esperado, para que la app web (`app.py`) sepa cómo preprocesar el dibujo del usuario antes de mandarlo al modelo.

In [ ]:
import json

model_config = {
    "img_size": IMG_SIZE,
    "class_names": class_names
}

with open("labels.json", "w") as f:
    json.dump(model_config, f, indent=2)

print("Archivo labels.json guardado")